# Other renderers 02: KCL

KCL is a typed configuration language with schemas, defaults and `check` blocks, and it renders YAML directly. `kcl vet` validates plain data files against a schema, `kcl test` runs `test_*` functions.


In [ ]:
export HOME=/tmp
mkdir -p /source/work/other-lab/kcl && cd /source/work/other-lab/kcl
cat > schema.k <<'KCL'
schema Context:
    env: "dev" | "prod"
    replicas: int = 1
    image: str = "traefik/whoami:v1.11.0"
    check:
        0 < replicas <= 20, "replicas must be between 1 and 20"
        not image.endswith(":latest"), "no floating tags"
KCL
cat > main.k <<'KCL'
import schema

prod = schema.Context {env = "prod", replicas = 3}

deployment = {
    apiVersion = "apps/v1"
    kind = "Deployment"
    metadata.name = "web"
    spec = {
        replicas = prod.replicas
        template.spec.containers = [{name = "web", image = prod.image}]
    }
}
KCL
kcl run main.k


In [ ]:
cd /source/work/other-lab/kcl
sed -i 's/replicas = 3/replicas = 30/' main.k && (kcl run main.k 2>&1 | grep -m2 -i 'replicas\|error') || true; sed -i 's/replicas = 30/replicas = 3/' main.k


In [ ]:
cd /source/work/other-lab/kcl
cat > data.yaml <<'YAML'
env: prod
replicas: 3
image: traefik/whoami:v1.11.0
YAML
kcl vet data.yaml schema.k --format yaml -d Context && echo "data.yaml conforms to Context"
kcl fmt main.k >/dev/null && kcl lint main.k && echo "fmt + lint ok"


In [ ]:
cd /source/work/other-lab/kcl
cat > main_test.k <<'KCL'
import schema

test_prod_replicas = lambda {
    ctx = schema.Context {env = "prod", replicas = 3}
    assert ctx.replicas == 3
}
KCL
kcl test ./... 2>&1 | tail -3
